# 🎯 LocateAnything-3B — Đánh giá độ chính xác trên COCO (Kaggle T4)

Đo **Precision / Recall / F1 @ IoU** + **MAE đếm** của `nvidia/LocateAnything-3B`
trên ảnh COCO val2017 có nhãn (person / car / bottle) — *điểm số thật* cho 3 bài
toán: đếm người, đếm xe, đếm sản phẩm.

### ⚙️ Điểm mấu chốt: KHÔNG cần cài transformers thủ công
Kaggle cài sẵn **transformers 5.0.0** nhưng model cần **4.57.1**. Quan trọng:
Kaggle **revert lại 5.0.0 mỗi lần restart kernel** → cách 'pip install + restart'
KHÔNG bao giờ ăn. Vì vậy `run_eval.py` **tự cài 4.57.1 rồi tự khởi động lại tiến
trình** (re-exec, không đụng kernel) — bạn chỉ việc Run All.


## 1) Tải code mới nhất


In [ ]:
%cd /kaggle/working
!rm -rf VisionOS
!git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git
%cd VisionOS/VisionOS
!git checkout -q claude/rebuild-visionos-codebase-tgg0mf
!git pull -q origin claude/rebuild-visionos-codebase-tgg0mf
print('--- commit đang dùng (phải có auto-pin) ---')
!git log --oneline -3


## 2) Kiểm chứng bộ chấm điểm (KHÔNG cần GPU, vài giây)
Xác nhận phần tính P/R/F1 + vẽ ảnh TP/FP/FN chạy đúng trước khi tốn GPU.


In [ ]:
!python run_eval.py --selftest


## 3) Phân tích dữ liệu đầu vào (KHÔNG cần model)
Xem mỗi lớp có bao nhiêu ảnh, vật to/nhỏ ra sao — để hiểu vì sao điểm cao/thấp.


In [ ]:
!python run_eval.py --analyze-only --classes person car bottle --n 50


## 4) 🚀 Chạy thử nhanh — LocateAnything trên 5 ảnh 'person'
**Cell này kích hoạt auto-pin.** Lần đầu sẽ thấy:
```
⚙️  transformers==5.0.0 KHÔNG khớp → cài 4.57.1 …
🔄 Khởi động lại với transformers==4.57.1 …
🔧 transformers==4.57.1 · torch==… · CUDA True · dtype=torch.float16
✅ Loaded in …s (device=cuda:0)
```
⏳ Lần đầu tốn ~1–2 phút (cài transformers + tải model 7.8GB). Kiên nhẫn nhé.


In [ ]:
# run_eval TỰ cài transformers==4.57.1 + tự re-exec (không cần restart kernel).
!python run_eval.py --model locate --classes person --n 5 --save-dir /kaggle/working/viz


## 5) 🏁 ĐIỂM SỐ THẬT — 3 lớp person / car / bottle
Chạy đầy đủ + lưu ảnh dự đoán (🟢 TP · 🔴 FP thừa · 🟡 GT bỏ sót) vào `/kaggle/working/viz`.
Tăng `--n` để số liệu ổn định hơn (ví dụ 30–50 ảnh/lớp).


In [ ]:
!python run_eval.py --model locate --classes person car bottle --n 30 --save-dir /kaggle/working/viz


## 6) Xem ảnh dự đoán để soi model sai ở đâu


In [ ]:
import glob
from IPython.display import Image, display
for cls in ['person', 'car', 'bottle']:
    paths = sorted(glob.glob(f'/kaggle/working/viz/{cls}/*.jpg'))[:4]
    print(f'=== {cls}: {len(paths)} ảnh mẫu ===')
    for p in paths:
        display(Image(p))


## 7) (Tùy chọn) So sánh với YOLOv8 (đã học sẵn COCO)
YOLO được huấn luyện thẳng trên COCO nên thường cao hơn — dùng làm mốc tham chiếu.
LocateAnything là **zero-shot open-vocab** (chưa từng học COCO) nên thấp hơn là bình thường.


In [ ]:
!python run_eval.py --model yolo --classes person car bottle --n 30


---
### Đọc kết quả
- **P (Precision)**: model báo có thì đúng bao nhiêu %. Thấp = nhiều báo nhầm (FP).
- **R (Recall)**: có bao nhiêu vật thật được tìm ra. Thấp = bỏ sót nhiều (FN).
- **F1**: điểm tổng hợp P và R. **MAE**: sai số đếm trung bình mỗi ảnh.
- Vật **càng nhỏ** (bottle 88% vật <1% diện tích) model open-vocab **càng khó** → F1 thấp là dễ hiểu.
- Muốn tăng điểm domain của bạn: đổi `prompt`, chỉnh `--dedup-iou`, hoặc **fine-tune**.
